# SageMaker Pipelines

## What are SageMaker Pipelines?

Pipelines automate ML workflows by defining steps for data processing, training, evaluation, and deployment. They integrate with the model registry for version control and enable continuous ML workflows.

## Creating a Basic Pipeline

In [ ]:
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.processing import ScriptProcessor
from sagemaker.estimator import Estimator
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::123456789012:role/SageMakerRole'
bucket = session.default_bucket()

# Define processing step
processor = ScriptProcessor(
    role=role,
    instance_type='ml.m5.xlarge',
    instance_count=1,
    framework_version='0.23-1',
    sagemaker_session=session
)

processing_step = ProcessingStep(
    name='ProcessingStep',
    processor=processor,
    code='preprocessing.py',
    inputs=[
        ProcessingInput(
            source=f's3://{bucket}/raw-data/',
            destination='/opt/ml/processing/input'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{bucket}/processed-data/'
        )
    ]
)

# Define training step
estimator = Estimator(
    image_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/training-output',
    sagemaker_session=session
)

training_step = TrainingStep(
    name='TrainingStep',
    estimator=estimator,
    inputs={'training': f's3://{bucket}/processed-data/'}
)

# Create pipeline
pipeline = Pipeline(
    name='ml-pipeline',
    parameters=[],
    steps=[processing_step, training_step]
)

# Execute pipeline
pipeline.upsert(role_arn=role)
pipeline.start()

## Pipeline Parameters

In [ ]:
from sagemaker.workflow.parameters import ParameterString, ParameterInteger

# Define parameters
instance_type = ParameterString(
    name='InstanceType',
    default_value='ml.m5.xlarge'
)

epochs = ParameterInteger(
    name='Epochs',
    default_value=10
)

# Use parameters in steps
estimator = Estimator(
    image_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    instance_count=1,
    instance_type=instance_type,
    output_path=f's3://{bucket}/training-output',
    sagemaker_session=session
)

estimator.set_hyperparameters(epochs=epochs)

## Conditional Steps

In [ ]:
from sagemaker.workflow.conditions import ConditionGreaterThan
from sagemaker.workflow.steps import ConditionStep

# Create condition
condition = ConditionGreaterThan(
    left=training_step.properties.FinalMetricDataList[0].Value,
    right=0.8
)

# Create conditional step
conditional_step = ConditionStep(
    name='ConditionalDeploymentStep',
    conditions=[condition],
    if_steps=[deployment_step],
    else_steps=[]
)

## Model Registry Integration

In [ ]:
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.model import Model

# Create model
model = Model(
    image_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    model_data=training_step.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    sagemaker_session=session
)

# Register model
from sagemaker.model_registry import ModelPackageGroup

model_package_group = ModelPackageGroup(
    name='xgboost-models',
    model_package_group_description='XGBoost models',
    sagemaker_session=session
)

model_package = model.register(
    model_package_group_name='xgboost-models',
    content_types=['text/csv'],
    response_types=['text/csv'],
    inference_instances=['ml.m5.large'],
    transform_instances=['ml.m5.large']
)

## Pipeline Configuration

```json
{
  "pipeline_config": {
    "pipeline_name": "ml-pipeline",
    "pipeline_definition": {
      "Version": "2020-12-01",
      "Metadata": {},
      "Parameters": [
        {
          "Name": "InstanceType",
          "Type": "String",
          "DefaultValue": "ml.m5.xlarge"
        }
      ],
      "Steps": [
        {
          "Name": "ProcessingStep",
          "Type": "Task",
          "Resource": "arn:aws:states:::sagemaker:createProcessingJob.sync"
        },
        {
          "Name": "TrainingStep",
          "Type": "Task",
          "Resource": "arn:aws:states:::sagemaker:createTrainingJob.sync"
        }
      ]
    }
  }
}
```

## Monitoring Pipeline Execution

In [ ]:
# Get pipeline execution details
execution = pipeline.start()
execution_arn = execution.arn

# Check execution status
import boto3

sm_client = boto3.client('sagemaker')
response = sm_client.describe_pipeline_execution(
    PipelineExecutionArn=execution_arn
)

print(f"Status: {response['PipelineExecutionStatus']}")
print(f"Start time: {response['CreationTime']}")

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is the primary purpose of SageMaker Pipelines?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="0">
      <span>Orchestrate end-to-end ML workflows</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="1">
      <span>Store training data</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="2">
      <span>Deploy models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="3">
      <span>Monitor endpoints</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What are pipeline parameters used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="0">
      <span>Storing model artifacts</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="1">
      <span>Monitoring performance</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="2">
      <span>Making pipelines reusable with different values</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="3">
      <span>Deploying models</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What do conditional steps enable?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="0">
      <span>Parallel execution</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="1">
      <span>Branching logic based on conditions</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="2">
      <span>Faster execution</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="3">
      <span>Cost reduction</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What is the model registry used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="0">
      <span>Version control and tracking of models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="1">
      <span>Training models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="2">
      <span>Processing data</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="3">
      <span>Monitoring endpoints</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ What enables MLOps in SageMaker?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="0">
      <span>Only training jobs</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="1">
      <span>Only endpoints</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="2">
      <span>Pipelines, model registry, and automation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="3">
      <span>Only monitoring</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>